# Fine-tune the Vietnamese humanizer (QLoRA, T4-class)

Train on **Colab or Kaggle free GPU (T4/P100, 16 GB)**. Five cells, run top to bottom:

1. GPU check + clone repo + install deps
2. Point at the human-written corpus
3. Build the SFT dataset (`prepare_dataset.py`)
4. Train QLoRA (`train_qlora.py`; auto-falls back to 7B if the 14B OOMs)
5. Merge the adapter + optional AWQ export; persist artifacts

Edit the CONFIG cell once. Regenerate this notebook by editing
`training/remote/mk_notebook.py` — the notebook itself is disposable.

In [ ]:
# ==== 0. CONFIG ====
REPO_URL = "https://github.com/datnguyentien2006/vietnamese-humanize.git"
DATA_RAW = None            # Colab: r"/content/drive/MyDrive/vietnamese-humanize/data-raw" (a folder of .txt/.txt.gz/.jsonl/.parquet corpora)
                           # Kaggle: leave None and attach ONE dataset with the text corpus — auto-detected
PERSIST_DIR = None         # Colab: r"/content/drive/MyDrive/vietnamese-humanize/out" ; on Kaggle outputs are kept automatically (kernel output)
TRAIN_MODEL = "Qwen/Qwen2.5-7B-Instruct"  # 7B: train nhanh gấp đôi, AWQ ~4,5 GB serve được trên GPU 6 GB của máy local.
                           # Đổi thành None để dùng 14B (config.yaml) — chỉ nên chọn khi có GPU ≥16 GB để serve.
LIMIT = None               # None -> all corpus docs; e.g. 20000 for a quick dry run
RUN_AWQ_EXPORT = True

In [ ]:
# ==== 1. GPU check + clone + install ====
!nvidia-smi

import os, subprocess

if not os.path.isdir("vietnamese-humanize/.git"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "vietnamese-humanize"], check=True)
%cd vietnamese-humanize

!pip install -q -r training/requirements.txt
!pip install -q huggingface_hub
!pip install -q autoawq   # bắt buộc cho --export-awq (merge_and_export import awq)
print("setup ok")

In [ ]:
# ==== 2. Corpus ====
# Chooses, in order: Kaggle attached dataset -> Colab Drive folder -> Colab prompt mount.

import glob, os, shutil


def use_corpus_from(source: str) -> None:
    if os.path.islink("training/data/raw"):
        os.remove("training/data/raw")
    elif os.path.isdir("training/data/raw"):
        shutil.rmtree("training/data/raw")
    os.makedirs("training/data", exist_ok=True)
    os.symlink(source, "training/data/raw")
    files = sorted(glob.glob(source + "*"))[:10]
    print("linked:", source)
    print("sample files:", files)


kaggle_inputs = sorted(glob.glob("/kaggle/input/*"))
if kaggle_inputs:
    corpus_root = kaggle_inputs[0]            # one dataset = the corpus
    use_corpus_from(corpus_root)
elif DATA_RAW:
    corpus_root = DATA_RAW
    use_corpus_from(corpus_root)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    corpus_root = input("Thư mục corpus (người viết) trong Drive: ")
    use_corpus_from(corpus_root)

# Prebuilt SFT (train/val.jsonl) nằm cùng dataset → copy sang sft/ để cell 3 bỏ qua build
sft_hits = glob.glob(corpus_root + "/**/train.jsonl", recursive=True) + glob.glob(
    corpus_root + "/**/val.jsonl", recursive=True
)
if sft_hits:
    os.makedirs("training/data/sft", exist_ok=True)
    for sft_file in sft_hits:
        shutil.copy(sft_file, "training/data/sft/")
    print("prebuilt SFT copied:", [os.path.basename(f) for f in sft_hits])
print("corpus ok")

In [ ]:
# ==== 3. Build SFT dataset (bỏ qua nếu đã có bản dựng sẵn) ====
import os

if os.path.isfile("training/data/sft/train.jsonl"):
    print("PREBUILT SFT FOUND — dùng train/val đã upload, bỏ qua prepare_dataset")
else:
    limit_arg = "" if LIMIT is None else f"--limit {LIMIT}"
    !python training/prepare_dataset.py {limit_arg}
with open("training/data/sft/train.jsonl", encoding="utf-8") as fh:
    print(fh.read(700))

In [ ]:
# ==== 4. Train QLoRA (14B, falls back to 7B on OOM) ====
import subprocess

FALLBACK = "Qwen/Qwen2.5-7B-Instruct"
TRAINED_MODEL = None


def try_train(model_id):
    global TRAINED_MODEL
    cmd = ["python", "-u", "training/train_qlora.py"]
    if model_id:
        cmd += ["--model", model_id]
    print("RUN:", *cmd)
    proc = subprocess.run(cmd)
    print("exit code:", proc.returncode)
    if proc.returncode == 0:
        TRAINED_MODEL = model_id
    return proc.returncode == 0


ok = try_train(TRAIN_MODEL)
if not ok and TRAIN_MODEL != FALLBACK:
    print("[warn] falling back to", FALLBACK)
    ok = try_train(FALLBACK)
assert ok, "Training failed on both 14B and 7B - lower LIMIT or free disk/VRAM, then retry."
print("adapter: training/output/humanizer-lora")
print("trained base:", TRAINED_MODEL)

In [ ]:
# ==== 5. Merge + AWQ export + persist ====
import os, shutil, subprocess

# Merge with the EXACT base that produced the adapter, and let accelerate
# shift layers between GPU/RAM (needed on small-RAM Colab hosts).
merge_cmd = ["python", "-u", "training/merge_and_export.py"]
if RUN_AWQ_EXPORT:
    merge_cmd.append("--export-awq")
base = TRAINED_MODEL or TRAIN_MODEL
if base:
    merge_cmd += ["--model", base]
# Ghi merged fp16 ra /tmp: /kaggle/working có hạn 20 GB, còn merged 7B (~15 GB)
# + AWQ (~4,5 GB) sẽ sát trần; /tmp không tính vào output.
merge_cmd += ["--merge-device", "auto", "--merged-dir", "/tmp/humanizer-merged"]
print("RUN:", *merge_cmd)
proc = subprocess.run(merge_cmd)
if proc.returncode != 0:
    # Adapter là artifact không thể tái tạo — KHÔNG raise để còn giữ nó.
    print("[warn] merge/AWQ lỗi — adapter vẫn được giữ ở training/output/humanizer-lora")
    print("[warn] có thể quantize lại sau bằng merge_and_export.py")
shutil.rmtree("/tmp/humanizer-merged", ignore_errors=True)
for _kind in ("lora", "awq"):
    _path = f"training/output/humanizer-{_kind}"
    if os.path.isdir(_path):
        _size = sum(
            os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(_path) for f in fs
        )
        print(f"{_kind}: {_size / 1e9:.2f} GB -> {_path}")

# Persist adapter + AWQ. The full fp16 merged model is skipped on purpose:
# it is ~15 GB, regenerable from adapter+base in minutes, and would blow the
# free Drive quota next to the AWQ copy.
if PERSIST_DIR:
    os.makedirs(PERSIST_DIR, exist_ok=True)
    for kind in ("lora", "awq"):
        src = f"training/output/humanizer-{kind}"
        dst = os.path.join(PERSIST_DIR, f"humanizer-{kind}")
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print("persisted:", dst)
        else:
            print("[skip] missing:", src)
else:
    print("artifacts remain in training/output/* for this session (download manually on Colab)")
print("PERSIST_DONE")